# Market Opportunity Classification

This notebook identifies which international markets are the best targets for Algeria by predicting a High / Medium / Low opportunity class for each `(Year, Importer, Product)` combination. The unit of analysis treats Algeria as the exporter and each importer-product pair as a potential market.

We build a forecasting task: features are from year $T$ and the label is the market opportunity class in year $T+1$. The model learns patterns in importer demand, Algeria's current market share, growth dynamics, and sector priority. Outputs include a ranked list of target markets saved to `market_opportunity_ranking.csv`.

## 1. Imports + file discovery

**Purpose and approach**
This cell discovers the required input files and selects the best available BACI dataset. The notebook is designed to run both on Kaggle and locally, so the file search is flexible and picks the largest match to avoid partial or outdated files.
By validating inputs early, we avoid long streaming runs that fail later due to missing data. This also keeps the pipeline reproducible by making file usage explicit.

In [ ]:
import os, gc, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:,.4f}'.format)

INPUT_ROOTS = [Path('/kaggle/input'), Path.cwd()]

def find_file(name, roots=INPUT_ROOTS):
    hits = []
    for root in roots:
        if root.exists():
            hits.extend(root.rglob(name))
    if not hits: return None
    return max(hits, key=lambda p: p.stat().st_size)

DATA_FILES = {n: find_file(n) for n in [
    'baci_features.csv', 'baci_clean.csv', '02_worldbank_features.csv',
]}

IS_KAGGLE = Path('/kaggle/working').exists()
BASE_OUT = Path('/kaggle/working') if IS_KAGGLE else Path.cwd()

if IS_KAGGLE:
    RESULTS_DIR = BASE_OUT
    MODEL_DIR = BASE_OUT
else:
    RESULTS_DIR = BASE_OUT / 'results'
    MODEL_DIR = BASE_OUT / 'model' / 'classification' / 'market_classification'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print('RESULTS_DIR =', RESULTS_DIR)
print('MODEL_DIR   =', MODEL_DIR)
for n, p in DATA_FILES.items():
    sz = f'{p.stat().st_size/1e9:>7.2f} GB' if p else 'MISSING'
    print(f'  {sz}  {n}  ->  {p}')

BACI_PATH = DATA_FILES['baci_clean.csv'] or DATA_FILES['baci_features.csv']
assert BACI_PATH is not None, 'Need baci_clean.csv or baci_features.csv'
print(f'\nUsing bilateral file: {BACI_PATH}')

OUT_DIR = /kaggle/working
    37.04 GB  baci_features.csv  ->  /kaggle/input/algeria-ml-export-dataset/content/baci_features.csv
    20.53 GB  baci_clean.csv  ->  /kaggle/input/algeria-ml-export-dataset/baci/baci_clean.csv
  MISSING  02_worldbank_features.csv  ->  None

Using bilateral file: /kaggle/input/algeria-ml-export-dataset/baci/baci_clean.csv


## 2. Constants

**Purpose and approach**
This cell defines the global constants used throughout the notebook: year range, priority sectors, and label thresholds. Keeping them centralized makes the modeling assumptions transparent and easy to adjust.
These thresholds translate economic intuition (big importer, low Algeria share, growing demand) into a consistent label definition.

In [4]:
YEAR_MIN, YEAR_MAX = 2012, 2023
PRIORITY_HS2 = {'07', '08', '15', '19', '21', '25', '28', '31', '68', '72'}
ALGERIA_ISO3 = 'DZA'
GROWTH_CLIP = 500.0
LABEL_MAP = {'Low': 0, 'Medium': 1, 'High': 2}
INV_LABEL = {v: k for k, v in LABEL_MAP.items()}
RANDOM_STATE = 42
TEST_LABEL_YEAR = 2023
TRAIN_MAX_LABEL_YEAR = 2022

# Score weights
W_IMP_PCT       = 0.40
W_IMP_GROWTH    = 0.35
W_LOW_ALG_SHARE = 0.10
W_PRIORITY      = 0.15

MARGIN_FRAC = 0.06
EXCLUDE_MARGIN = True
FILTER_TO_ALGERIA_PRODUCTS = True

# Model flags — all enabled to maximise ensemble diversity
RUN_TUNING = True           # XGBoost hyperparameter search (~10-15 min extra)
RUN_RF = True; RUN_XGB = True; RUN_LGBM = True
RUN_CATBOOST = True         # native categorical handling for Importer × HS2
RUN_SVM = True; RUN_MLP = True

MLP_LABEL_SMOOTHING = 0.1   # for the Keras MLP, attacks the Medium boundary

## 3. GPU check

**Purpose and approach**
This cell checks GPU availability and configures training devices. If a GPU is present, XGBoost and LightGBM are accelerated; otherwise they fall back to CPU.
We keep this logic explicit so results are reproducible across environments without changing the modeling pipeline.

In [5]:
import subprocess
try:
    nvsmi = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                            capture_output=True, text=True, timeout=5)
    print('nvidia-smi:', nvsmi.stdout.strip() or '(no GPU)')
except Exception as e:
    print('nvidia-smi not available:', e)
GPU_AVAILABLE = False
try:
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    GPU_AVAILABLE = bool(gpus)
    for g in gpus:
        try: tf.config.experimental.set_memory_growth(g, True)
        except Exception: pass
    print('TF GPUs:', gpus)
except Exception as e:
    print('TF import failed:', e)
XGB_DEVICE = 'cuda' if GPU_AVAILABLE else 'cpu'
LGB_DEVICE = 'gpu'  if GPU_AVAILABLE else 'cpu'
CB_TASK = 'GPU' if GPU_AVAILABLE else 'CPU'
print(f'GPU={GPU_AVAILABLE}  XGB={XGB_DEVICE}  LGB={LGB_DEVICE}  CatBoost={CB_TASK}')

nvidia-smi: Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


2026-05-25 12:51:13.884276: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779713474.106680      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779713474.171949      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779713474.674909      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779713474.674964      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779713474.674968      57 computation_placer.cc:177] computation placer alr

TF GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
GPU=True  XGB=cuda  LGB=gpu  CatBoost=GPU


## 4. Geographic / linguistic dictionaries for Algeria-Importer bilateral signal

Hardcoded country groupings — these are the bilateral signals that current features lack. Most powerful additions per the diagnosis:

- **`is_neighbour`** — Algeria's immediate land borders or sea-adjacent (TUN, LBY, MAR, NER, MLI, MRT). Low logistics cost.
- **`is_mediterranean`** — Mediterranean basin. Cheap shipping.
- **`is_francophone`** — French is official/widely spoken. Cultural / business proximity (Algeria's second language).
- **`is_arabophone`** — Arabic is official. Cultural / business proximity (Algeria's official language).
- **`is_eu`** — European Union member. Algeria has a EU association agreement, lower trade barriers.
- **`distance_band`** — ordinal 1 (near) / 2 (medium) / 3 (far) based on geographic distance from Algeria.
- **`geo_region`** — ISO3 → world region (target-encoded later).

**Purpose and approach**
This cell defines geographic and linguistic groupings for importer countries. These features capture real-world trade frictions and affinities that are not visible in raw trade values alone.
We encode proximity (neighbors, Mediterranean), cultural/linguistic ties (francophone, arabophone), policy proximity (EU), and a coarse distance band. These signals help the model distinguish markets that are structurally easier for Algeria to enter.

In [6]:
NEIGHBOURS = {'TUN', 'LBY', 'MAR', 'NER', 'MLI', 'MRT'}
MEDITERRANEAN = {'FRA','ESP','ITA','GRC','MLT','CYP','SVN','HRV','TUR',
                 'TUN','LBY','EGY','MAR','ISR','LBN','SYR','MNE','ALB','BIH'}
FRANCOPHONE = {'FRA','BEL','LUX','CHE','CAN','MCO',
               'TUN','MAR','SEN','CIV','BFA','MLI','NER','TCD','CMR',
               'GAB','COG','COD','BEN','TGO','MDG','COM','DJI','GIN','CAF',
               'RWA','BDI','HTI','LBN','VUT','SYC','MUS'}
ARABOPHONE = {'EGY','TUN','LBY','MAR','SAU','ARE','KWT','QAT','OMN','BHR',
              'JOR','LBN','SYR','IRQ','YEM','MRT','SDN','SOM','DJI','PSE','COM'}
EU_MEMBERS = {'AUT','BEL','BGR','HRV','CYP','CZE','DNK','EST','FIN','FRA','DEU',
              'GRC','HUN','IRL','ITA','LVA','LTU','LUX','MLT','NLD','POL','PRT',
              'ROU','SVK','SVN','ESP','SWE'}

DIST_NEAR  = NEIGHBOURS | {'EGY'}
DIST_MED   = MEDITERRANEAN | {'PRT','GBR','IRL','DEU','BEL','NLD','SAU','ARE','TUR',
                              'SEN','CIV','GHA','CMR','NGA','MRT','SDN','JOR','LBN'}
def distance_band(iso):
    if iso in DIST_NEAR: return 1
    if iso in DIST_MED:  return 2
    return 3

REGION_MAP = {
    # North Africa
    'TUN':'north_africa','LBY':'north_africa','MAR':'north_africa','EGY':'north_africa','MRT':'north_africa','SDN':'north_africa',
    # Sub-Saharan Africa
    'NGA':'ssa','ZAF':'ssa','KEN':'ssa','GHA':'ssa','CIV':'ssa','SEN':'ssa','ETH':'ssa','TZA':'ssa',
    'UGA':'ssa','CMR':'ssa','AGO':'ssa','COD':'ssa','CGO':'ssa','GIN':'ssa','MLI':'ssa','NER':'ssa',
    'BFA':'ssa','TCD':'ssa','BEN':'ssa','TGO':'ssa','GAB':'ssa','MDG':'ssa','RWA':'ssa','BDI':'ssa',
    'NAM':'ssa','BWA':'ssa','MOZ':'ssa','ZWE':'ssa','ZMB':'ssa','MWI':'ssa','MUS':'ssa','SYC':'ssa',
    # Western / Northern Europe
    'FRA':'west_eu','GBR':'west_eu','ESP':'west_eu','ITA':'west_eu','NLD':'west_eu','BEL':'west_eu',
    'DEU':'west_eu','CHE':'west_eu','AUT':'west_eu','PRT':'west_eu','IRL':'west_eu','GRC':'west_eu',
    'DNK':'west_eu','SWE':'west_eu','NOR':'west_eu','FIN':'west_eu','LUX':'west_eu','ISL':'west_eu',
    # Eastern Europe
    'POL':'east_eu','CZE':'east_eu','HUN':'east_eu','ROU':'east_eu','SVK':'east_eu','RUS':'east_eu',
    'UKR':'east_eu','BGR':'east_eu','HRV':'east_eu','SVN':'east_eu','EST':'east_eu','LVA':'east_eu',
    'LTU':'east_eu','SRB':'east_eu','BLR':'east_eu','ALB':'east_eu','MNE':'east_eu','BIH':'east_eu',
    'MKD':'east_eu','MDA':'east_eu',
    # Middle East
    'SAU':'middle_east','ARE':'middle_east','QAT':'middle_east','KWT':'middle_east','OMN':'middle_east',
    'BHR':'middle_east','JOR':'middle_east','LBN':'middle_east','IRQ':'middle_east','ISR':'middle_east',
    'TUR':'middle_east','IRN':'middle_east','YEM':'middle_east','SYR':'middle_east','PSE':'middle_east',
    # North America
    'USA':'north_am','CAN':'north_am','MEX':'north_am',
    # Latin America
    'BRA':'latin_am','ARG':'latin_am','CHL':'latin_am','COL':'latin_am','PER':'latin_am','VEN':'latin_am',
    'URY':'latin_am','PRY':'latin_am','BOL':'latin_am','ECU':'latin_am','CUB':'latin_am','GTM':'latin_am',
    'PAN':'latin_am','CRI':'latin_am','DOM':'latin_am','HND':'latin_am','SLV':'latin_am','NIC':'latin_am',
    'JAM':'latin_am','TTO':'latin_am','HTI':'latin_am',
    # East Asia
    'CHN':'east_asia','JPN':'east_asia','KOR':'east_asia','TWN':'east_asia','HKG':'east_asia','MAC':'east_asia',
    # South Asia
    'IND':'south_asia','PAK':'south_asia','BGD':'south_asia','LKA':'south_asia','NPL':'south_asia','AFG':'south_asia',
    # Southeast Asia
    'SGP':'se_asia','VNM':'se_asia','THA':'se_asia','MYS':'se_asia','IDN':'se_asia','PHL':'se_asia',
    'MMR':'se_asia','KHM':'se_asia','LAO':'se_asia','BRN':'se_asia',
    # Oceania
    'AUS':'oceania','NZL':'oceania','FJI':'oceania','PNG':'oceania',
}
print('Region mappings:', len(REGION_MAP), 'countries')
print('Mediterranean:', len(MEDITERRANEAN), '| EU:', len(EU_MEMBERS),
      '| Francophone:', len(FRANCOPHONE), '| Arabophone:', len(ARABOPHONE))

Region mappings: 141 countries
Mediterranean: 19 | EU: 27 | Francophone: 32 | Arabophone: 21


## 5. Stream the bilateral BACI file

Single pass through ~140M rows. Two accumulators: Algeria's outgoing flows, and per-`(Year, Importer, Product)` demand.

**Purpose and approach**
This cell streams the bilateral file in chunks and accumulates two partial tables: Algeria's flows and importer demand by `(Year, Importer, Product)`.
It avoids loading the full dataset into memory and prints progress so we can monitor coverage and performance.

In [7]:
NEEDED = ['Year', 'Exporter_ISO3', 'Importer_ISO3', 'Product', 'Value']
DTYPES = {
    'Year': 'int16', 'Exporter_ISO3': 'category', 'Importer_ISO3': 'category',
    'Product': 'string', 'Value': 'float32',
}
CHUNK = 500_000
Y_LOAD_MIN = YEAR_MIN - 2

alg_chunks = []
demand_chunks = []
n_rows = 0
for i, chunk in enumerate(pd.read_csv(BACI_PATH, chunksize=CHUNK,
                                      usecols=lambda c: c in NEEDED, dtype=DTYPES)):
    n_rows += len(chunk)
    chunk = chunk[(chunk['Year'] >= Y_LOAD_MIN) & (chunk['Year'] <= YEAR_MAX)]
    if not len(chunk): continue
    alg = chunk[chunk['Exporter_ISO3'].astype(str) == ALGERIA_ISO3]
    if len(alg):
        alg_chunks.append(alg[['Year', 'Importer_ISO3', 'Product', 'Value']].copy())
    g = chunk.groupby(['Year', 'Importer_ISO3', 'Product'], observed=True)['Value'].sum().reset_index()
    demand_chunks.append(g)
    if i % 40 == 0:
        kept_alg = sum(len(c) for c in alg_chunks)
        kept_dem = sum(len(c) for c in demand_chunks)
        print(f'  chunk {i:>4d}  raw={n_rows:>12,}  algeria={kept_alg:>9,}  demand_partials={kept_dem:>10,}')
    del chunk
    gc.collect()

print(f'\nDone. Raw: {n_rows:,}  algeria: {sum(len(c) for c in alg_chunks):,}  demand_partials: {sum(len(c) for c in demand_chunks):,}')

  chunk    0  raw=     500,000  algeria=    4,094  demand_partials=   306,198
  chunk   40  raw=  20,500,000  algeria=   13,261  demand_partials=13,564,599
  chunk   80  raw=  40,500,000  algeria=   23,994  demand_partials=27,105,335
  chunk  120  raw=  60,500,000  algeria=   30,743  demand_partials=40,600,798
  chunk  160  raw=  80,500,000  algeria=   40,632  demand_partials=54,294,746
  chunk  200  raw= 100,500,000  algeria=   52,534  demand_partials=67,992,514
  chunk  240  raw= 120,500,000  algeria=   66,417  demand_partials=81,530,443

Done. Raw: 142,112,452  algeria: 66,417  demand_partials: 88,679,542


## 6. Collapse partials + compute Algeria-side aggregates

Four aggregates derived from the streaming pass:
1. `algeria_flows` — Algeria's per-(Year, Importer, Product) exports
2. `algeria_world_per_product` — Algeria's total world exports per product (capability proxy)
3. `algeria_total_per_importer` — Algeria's total trade with each importer (relationship proxy)
4. `algeria_per_imp_hs2` — Algeria's total exports per (Year, Importer, HS2 chapter) to capture sector presence in each market

**Purpose and approach**
This cell collapses the chunk-level partials into final aggregate tables and derives a list of products Algeria has exported.
These aggregates become the building blocks for the main panel and allow us to compute market shares and capability signals.

In [8]:
algeria_flows = (pd.concat(alg_chunks, ignore_index=True)
                 .groupby(['Year', 'Importer_ISO3', 'Product'], observed=True, as_index=False)['Value']
                 .sum()
                 .rename(columns={'Value': 'Algeria_Value'}))
algeria_flows['Algeria_Value'] = algeria_flows['Algeria_Value'].astype('float32')
algeria_flows['HS2_tmp'] = algeria_flows['Product'].astype(str).str.zfill(6).str[:2]
del alg_chunks; gc.collect()
print('algeria_flows  :', algeria_flows.shape)

algeria_world_per_product = (algeria_flows
    .groupby(['Year', 'Product'], observed=True, as_index=False)['Algeria_Value']
    .sum().rename(columns={'Algeria_Value': 'Algeria_World_Exports'}))
algeria_world_per_product['Algeria_World_Exports'] = algeria_world_per_product['Algeria_World_Exports'].astype('float32')
print('algeria_world_per_product:', algeria_world_per_product.shape)

algeria_total_per_importer = (algeria_flows
    .groupby(['Year', 'Importer_ISO3'], observed=True, as_index=False)['Algeria_Value']
    .sum().rename(columns={'Algeria_Value': 'Algeria_Total_to_Importer'}))
algeria_total_per_importer['Algeria_Total_to_Importer'] = algeria_total_per_importer['Algeria_Total_to_Importer'].astype('float32')
print('algeria_total_per_importer:', algeria_total_per_importer.shape)

algeria_per_imp_hs2 = (algeria_flows
    .groupby(['Year', 'Importer_ISO3', 'HS2_tmp'], observed=True, as_index=False)['Algeria_Value']
    .sum().rename(columns={'Algeria_Value': 'Algeria_HS2_to_Importer', 'HS2_tmp': 'HS2'}))
algeria_per_imp_hs2['Algeria_HS2_to_Importer'] = algeria_per_imp_hs2['Algeria_HS2_to_Importer'].astype('float32')
print('algeria_per_imp_hs2:', algeria_per_imp_hs2.shape)

importer_demand = (pd.concat(demand_chunks, ignore_index=True)
                   .groupby(['Year', 'Importer_ISO3', 'Product'], observed=True, as_index=False)['Value']
                   .sum()
                   .rename(columns={'Value': 'Importer_Demand'}))
importer_demand['Importer_Demand'] = importer_demand['Importer_Demand'].astype('float32')
del demand_chunks; gc.collect()
print('importer_demand:', importer_demand.shape)

algeria_products = set(
    algeria_world_per_product[algeria_world_per_product['Algeria_World_Exports'] > 0]['Product'].astype(str)
 )
print(f'\nAlgeria has exported {len(algeria_products):,} distinct products in 2012-2023')

algeria_flows  : (66417, 5)
algeria_world_per_product: (21364, 3)
algeria_total_per_importer: (1669, 3)
algeria_per_imp_hs2: (22854, 4)
importer_demand: (10331079, 4)

Algeria has exported 3,735 distinct products in 2012-2023


## 7. Build the panel + derived features + filters

Merges importer demand, Algeria's flows, Algeria's per-(Year, Product) capacity, Algeria's per-(Year, Importer) trade volume, Algeria's per-(Year, Importer, HS2) sector presence, global demand, and finally **geographic features** for each importer.

**Purpose and approach**
This cell merges the aggregate tables into a full panel, computes ratios like importer demand share, and adds geographic and relationship features.
It also filters invalid rows (zero demand, special codes) and optionally limits to products Algeria has already exported to keep the task realistic.

In [9]:
# Normalise key dtypes
for k in ['Year', 'Importer_ISO3', 'Product']:
    if k == 'Year':
        importer_demand[k] = importer_demand[k].astype('int16')
        algeria_flows[k]   = algeria_flows[k].astype('int16')
    else:
        importer_demand[k] = importer_demand[k].astype(str)
        algeria_flows[k]   = algeria_flows[k].astype(str)

df = importer_demand.merge(algeria_flows[['Year','Importer_ISO3','Product','Algeria_Value']],
                            on=['Year','Importer_ISO3','Product'], how='left')
df['Algeria_Value'] = df['Algeria_Value'].fillna(0).astype('float32')
df['HS2'] = df['Product'].str.zfill(6).str[:2].astype('category')
df['in_priority_hs2'] = df['HS2'].astype(str).isin(PRIORITY_HS2).astype('int8')
del importer_demand, algeria_flows; gc.collect()

algeria_world_per_product['Year'] = algeria_world_per_product['Year'].astype('int16')
algeria_world_per_product['Product'] = algeria_world_per_product['Product'].astype(str)
df = df.merge(algeria_world_per_product, on=['Year','Product'], how='left')
df['Algeria_World_Exports'] = df['Algeria_World_Exports'].fillna(0).astype('float32')

algeria_total_per_importer['Year'] = algeria_total_per_importer['Year'].astype('int16')
algeria_total_per_importer['Importer_ISO3'] = algeria_total_per_importer['Importer_ISO3'].astype(str)
df = df.merge(algeria_total_per_importer, on=['Year','Importer_ISO3'], how='left')
df['Algeria_Total_to_Importer'] = df['Algeria_Total_to_Importer'].fillna(0).astype('float32')
df['has_trade_relationship'] = (df['Algeria_Total_to_Importer'] > 0).astype('int8')

algeria_per_imp_hs2['Year'] = algeria_per_imp_hs2['Year'].astype('int16')
algeria_per_imp_hs2['Importer_ISO3'] = algeria_per_imp_hs2['Importer_ISO3'].astype(str)
algeria_per_imp_hs2['HS2'] = algeria_per_imp_hs2['HS2'].astype(str)
df['HS2_str'] = df['HS2'].astype(str)
df = df.merge(algeria_per_imp_hs2, left_on=['Year','Importer_ISO3','HS2_str'],
              right_on=['Year','Importer_ISO3','HS2'], how='left', suffixes=('', '_dropme'))
df = df.drop(columns=[c for c in df.columns if c.endswith('_dropme')])
df['Algeria_HS2_to_Importer'] = df['Algeria_HS2_to_Importer'].fillna(0).astype('float32')
df['algeria_active_in_imp_hs2'] = (df['Algeria_HS2_to_Importer'] > 0).astype('int8')
df = df.drop(columns=['HS2_str'])

wd = (df.groupby(['Year', 'Product'], observed=True, as_index=False)['Importer_Demand']
        .sum().rename(columns={'Importer_Demand': 'Global_Demand'}))
wd['Global_Demand'] = wd['Global_Demand'].astype('float32')
df = df.merge(wd, on=['Year', 'Product'], how='left')
del wd; gc.collect()

df['Importer_Demand_Pct'] = (df['Importer_Demand'] / df['Global_Demand'].replace(0, np.nan) * 100).astype('float32')
df['Algeria_Share_in_Importer'] = (df['Algeria_Value'] / df['Importer_Demand'].replace(0, np.nan) * 100).astype('float32')
df['Algeria_HS2_share_in_importer'] = (df['Algeria_HS2_to_Importer'] / df['Importer_Demand'].replace(0, np.nan) * 100).astype('float32')

df['is_neighbour']    = df['Importer_ISO3'].isin(NEIGHBOURS).astype('int8')
df['is_mediterranean']= df['Importer_ISO3'].isin(MEDITERRANEAN).astype('int8')
df['is_francophone']  = df['Importer_ISO3'].isin(FRANCOPHONE).astype('int8')
df['is_arabophone']   = df['Importer_ISO3'].isin(ARABOPHONE).astype('int8')
df['is_eu']           = df['Importer_ISO3'].isin(EU_MEMBERS).astype('int8')
df['distance_band']   = df['Importer_ISO3'].apply(distance_band).astype('int8')
df['geo_region']      = df['Importer_ISO3'].map(REGION_MAP).fillna('other').astype('category')

df = df[df['Importer_Demand'] > 0].reset_index(drop=True)
df = df[~df['Importer_ISO3'].str.startswith('S')].reset_index(drop=True)

if FILTER_TO_ALGERIA_PRODUCTS:
    n_before = len(df)
    df = df[df['Product'].isin(algeria_products)].reset_index(drop=True)
    print(f'Algeria-products filter: {n_before:,} -> {len(df):,} rows ({len(df)/n_before*100:.1f}% retained)')

print('\nFinal panel:', df.shape, ' memory:', f'{df.memory_usage(deep=True).sum()/1e9:.2f} GB')
print('Unique importers:', df['Importer_ISO3'].nunique(), '  unique products:', df['Product'].nunique())
print('\nGeographic feature distribution:')
for c in ['is_neighbour','is_mediterranean','is_francophone','is_arabophone','is_eu']:
    print(f'  {c}: {df[c].mean()*100:.1f}% rows = 1')
print('distance_band:', df['distance_band'].value_counts().to_dict())
print('geo_region:', df['geo_region'].value_counts().head(10).to_dict())

Algeria-products filter: 9,344,944 -> 7,391,783 rows (79.1% retained)

Final panel: (7391783, 23)  memory: 1.15 GB
Unique importers: 204   unique products: 3735

Geographic feature distribution:
  is_neighbour: 3.1% rows = 1
  is_mediterranean: 9.9% rows = 1
  is_francophone: 14.7% rows = 1
  is_arabophone: 9.1% rows = 1
  is_eu: 14.4% rows = 1
distance_band: {3: 6036607, 2: 1084971, 1: 270205}
geo_region: {'other': 2050350, 'ssa': 1139462, 'latin_am': 822359, 'west_eu': 755344, 'east_eu': 735057, 'middle_east': 540545, 'se_asia': 383663, 'south_asia': 248624, 'east_asia': 213891, 'north_africa': 201274}


## 8. Per-series growth + market opportunity score

**Purpose and approach**
This cell computes growth rates and constructs the market opportunity score from ranked components: importer size, growth, low Algeria share, and priority sector.
The score is a continuous signal used to define tercile labels and provides an interpretable basis for opportunity.

In [10]:
df = df.sort_values(['Importer_ISO3', 'Product', 'Year']).reset_index(drop=True)
g = df.groupby(['Importer_ISO3', 'Product'], observed=True)

df['Importer_Demand_Growth'] = (g['Importer_Demand'].pct_change() * 100).astype('float32')
df['Algeria_Value_Growth']   = (g['Algeria_Value'].pct_change()   * 100).astype('float32')

def market_opp_score(frame):
    yr = frame['Year']
    pct_rank = frame['Importer_Demand_Pct'].fillna(0).groupby(yr).rank(pct=True, method='average').astype('float32')
    low_share = (100 - frame['Algeria_Share_in_Importer'].fillna(0).clip(0, 100))
    low_share_rank = low_share.groupby(yr).rank(pct=True, method='average').astype('float32')
    growth_clipped = frame['Importer_Demand_Growth'].fillna(0).clip(-GROWTH_CLIP, GROWTH_CLIP)
    growth_rank = growth_clipped.groupby(yr).rank(pct=True, method='average').astype('float32')
    priority = frame['in_priority_hs2'].astype('float32')
    score = (W_IMP_PCT       * pct_rank +
             W_LOW_ALG_SHARE * low_share_rank +
             W_IMP_GROWTH    * growth_rank +
             W_PRIORITY      * priority)
    return score.astype('float32')

df['market_opp_score'] = market_opp_score(df)
print('Score stats per year:')
print(df.groupby('Year')['market_opp_score'].describe()[['mean','std','min','max']].round(4))

Score stats per year:
       mean    std    min    max
Year                            
2012 0.4436 0.1250 0.1750 0.7753
2013 0.4437 0.1699 0.0021 0.9384
2014 0.4439 0.1691 0.0279 0.9389
2015 0.4441 0.1688 0.0120 0.9398
2016 0.4441 0.1696 0.0270 0.9398
2017 0.4442 0.1714 0.0070 0.9392
2018 0.4442 0.1708 0.0084 0.9395
2019 0.4443 0.1692 0.0119 0.9401
2020 0.4443 0.1727 0.0015 0.9411
2021 0.4442 0.1720 0.0238 0.9388
2022 0.4443 0.1696 0.0162 0.9387
2023 0.4443 0.1666 0.0111 0.9391


## 9. Label assignment (per-year tercile at T+1 with margin)

**Purpose and approach**
This cell assigns class labels by taking per-year terciles of the next-year score and marks boundary samples with a margin flag.
Boundary exclusion removes ambiguous rows from training while keeping the full test year for evaluation.

In [11]:
def label_with_margin(score_col, year_col, margin_frac=MARGIN_FRAC):
    labels = pd.Series(np.ones(len(score_col), dtype='int8'), index=score_col.index)
    in_margin = pd.Series(np.zeros(len(score_col), dtype=bool), index=score_col.index)
    for _, idx in score_col.groupby(year_col).groups.items():
        s = score_col.loc[idx]
        p33 = s.quantile(1/3); p67 = s.quantile(2/3)
        margin = (p67 - p33) * margin_frac
        labels.loc[idx] = np.where(s < p33, 0, np.where(s > p67, 2, 1)).astype('int8')
        in_margin.loc[idx] = (((s > p33 - margin) & (s < p33 + margin)) |
                              ((s > p67 - margin) & (s < p67 + margin)))
    return labels, in_margin

df['label_thisyear'], _ = label_with_margin(df['market_opp_score'], df['Year'])
g = df.groupby(['Importer_ISO3', 'Product'], observed=True)
df['market_opp_score_next'] = g['market_opp_score'].shift(-1).astype('float32')
df['label_year'] = (df['Year'] + 1).astype('int16')
df = df.dropna(subset=['market_opp_score_next']).reset_index(drop=True)
df['label'], df['in_margin'] = label_with_margin(df['market_opp_score_next'], df['label_year'])

print('Shape:', df.shape)
print('Label distribution (T+1 tercile):')
print(df['label'].map(INV_LABEL).value_counts())
print(f'Margin samples: {df["in_margin"].sum():,} ({df["in_margin"].mean()*100:.1f}%)')

Shape: (6677951, 31)
Label distribution (T+1 tercile):
label
High      2225985
Medium    2225984
Low       2225982
Name: count, dtype: int64
Margin samples: 495,637 (7.4%)


## 10. Lag features + 3-year rolling stats

**Purpose and approach**
This cell creates lag features and rolling statistics for each importer-product series (lags, 3-year mean, 3-year std).
These features capture momentum and stability, which are critical for forecasting next-year opportunity.

In [12]:
g = df.groupby(['Importer_ISO3', 'Product'], observed=True)

for c in ['Importer_Demand', 'Importer_Demand_Pct', 'Importer_Demand_Growth',
         'Algeria_Value', 'Algeria_Share_in_Importer', 'market_opp_score',
         'Algeria_HS2_to_Importer']:
    df[f'{c}_lag1'] = g[c].shift(1).astype('float32')
    df[f'{c}_lag2'] = g[c].shift(2).astype('float32')

for c in ['Algeria_World_Exports', 'Algeria_Total_to_Importer']:
    df[f'{c}_lag1'] = g[c].shift(1).astype('float32')

for c in ['Importer_Demand', 'Importer_Demand_Growth',
         'Algeria_Share_in_Importer', 'market_opp_score']:
    df[f'{c}_3y_mean'] = g[c].transform(lambda s: s.rolling(3, min_periods=1).mean()).astype('float32')
    df[f'{c}_3y_std']  = g[c].transform(lambda s: s.rolling(3, min_periods=2).std()).astype('float32')

df['label_lag1'] = g['label_thisyear'].shift(1).fillna(-1).astype('int8')
df['market_opp_score_diff_2y'] = (df['market_opp_score'] - df['market_opp_score_lag2']).astype('float32')

print('After lags + rolling:', df.shape, ' memory:', f'{df.memory_usage(deep=True).sum()/1e9:.2f} GB')

After lags + rolling: (6677951, 57)  memory: 1.86 GB


## 11. Target encoding (HS2, Importer, joint, region) — train rows only

**Purpose and approach**
This cell computes target encodings for HS2, importer, region, and importer-by-HS2 using training years only.
Target encoding summarizes categorical effects without one-hot expansion and avoids leakage from the test year.

In [13]:
train_mask_enc = df['label_year'].values <= TRAIN_MAX_LABEL_YEAR
global_mean = float(df.loc[train_mask_enc, 'label'].mean())

def target_encode(group_col):
    m = df.loc[train_mask_enc].groupby(group_col, observed=True)['label'].mean()
    return df[group_col].map(m).astype('float32').fillna(global_mean)

df['hs2_target_enc']    = target_encode('HS2')
df['imp_target_enc']    = target_encode('Importer_ISO3')
df['region_target_enc'] = target_encode('geo_region')

# Joint (Importer, HS2)
imp_hs2_tr = (df.loc[train_mask_enc].groupby(['Importer_ISO3','HS2'], observed=True)['label'].mean()).to_dict()
mi = pd.MultiIndex.from_arrays([df['Importer_ISO3'].astype(str), df['HS2'].astype(str)])
df['imp_hs2_target_enc'] = mi.map(imp_hs2_tr.get).astype('float32').fillna(global_mean)

print('global mean (train):', round(global_mean, 4))
for c in ['hs2_target_enc','imp_target_enc','region_target_enc','imp_hs2_target_enc']:
    print(f'  {c}: {df[c].min():.4f} - {df[c].max():.4f}')
gc.collect()

global mean (train): 1.0
  hs2_target_enc: 0.7362 - 1.6348
  imp_target_enc: 0.2461 - 1.8914
  region_target_enc: 0.6068 - 1.7696
  imp_hs2_target_enc: 0.0000 - 2.0000


0

## 12. Global demand growth + supplier competition

**Purpose and approach**
This cell adds global context signals such as demand growth and supplier competition. These features indicate whether a market is expanding and how crowded the exporter landscape is.
They help the model differentiate between local fluctuations and genuine global demand shifts.

In [14]:
gd = (df.groupby(['Product', 'Year'], observed=True)['Global_Demand']
        .first().reset_index().sort_values(['Product', 'Year']))
gd['global_demand_growth'] = (gd.groupby('Product')['Global_Demand'].pct_change() * 100).astype('float32')
df = df.merge(gd[['Product','Year','global_demand_growth']], on=['Product','Year'], how='left')

imp_count = (df.groupby(['Product','Year'], observed=True)['Importer_ISO3']
             .nunique().reset_index().rename(columns={'Importer_ISO3':'n_importers'}))
imp_count['n_importers'] = imp_count['n_importers'].astype('int16')
df = df.merge(imp_count, on=['Product','Year'], how='left')
del gd, imp_count; gc.collect()
print('Added global_demand_growth and n_importers')

Added global_demand_growth and n_importers


## 13. Feature matrix

**Purpose and approach**
This cell builds the final modeling table used by all models. It combines numeric signals, HS2 features, and any optional macro variables into a single dataset.
Using one consistent feature table ensures model results are comparable and interpretable.

In [15]:
def safe_log1p(s): return np.log1p(np.clip(s.astype('float32'), 0, None)).astype('float32')

for col in ['Importer_Demand', 'Algeria_Value', 'Global_Demand',
            'Importer_Demand_lag1', 'Algeria_Value_lag1', 'Importer_Demand_lag2',
            'Algeria_World_Exports', 'Algeria_World_Exports_lag1',
            'Algeria_Total_to_Importer', 'Algeria_Total_to_Importer_lag1',
            'Algeria_HS2_to_Importer', 'Algeria_HS2_to_Importer_lag1']:
    if col in df.columns:
        df[f'log_{col}'] = safe_log1p(df[col])

df['imp_pct_x_growth'] = (df['Importer_Demand_Pct'].fillna(0) *
                          df['Importer_Demand_Growth'].fillna(0).clip(-200, 200)).astype('float32')

imp_macro_cols = [c for c in df.columns if c.startswith('imp_NY.') or c.startswith('imp_SP.')
                                          or c.startswith('imp_TG.') or c.startswith('imp_NE.')
                                          or c.startswith('imp_gdp_') or c.startswith('imp_trade_')
                                          or c.startswith('imp_import_')]

FEATURE_COLS = [
    # Autocorrelation backbone
    'label_thisyear', 'label_lag1', 'market_opp_score',
    # Year-T snapshot
    'log_Importer_Demand', 'Importer_Demand_Pct', 'Importer_Demand_Growth',
    'log_Algeria_Value', 'Algeria_Share_in_Importer', 'Algeria_Value_Growth',
    'log_Global_Demand',
    # Algeria capability + relationship
    'log_Algeria_World_Exports', 'log_Algeria_World_Exports_lag1',
    'log_Algeria_Total_to_Importer', 'log_Algeria_Total_to_Importer_lag1',
    'has_trade_relationship',
    # Algeria sector presence in this importer
    'log_Algeria_HS2_to_Importer', 'log_Algeria_HS2_to_Importer_lag1',
    'Algeria_HS2_share_in_importer', 'algeria_active_in_imp_hs2',
    # Geographic / linguistic / political
    'is_neighbour', 'is_mediterranean', 'is_francophone', 'is_arabophone',
    'is_eu', 'distance_band', 'region_target_enc',
    # Lags
    'Importer_Demand_lag1', 'Importer_Demand_Pct_lag1', 'Importer_Demand_Growth_lag1',
    'Algeria_Value_lag1', 'Algeria_Share_in_Importer_lag1', 'market_opp_score_lag1',
    'Importer_Demand_lag2', 'market_opp_score_lag2',
    # Rolling stats
    'Importer_Demand_3y_mean', 'Importer_Demand_3y_std',
    'Importer_Demand_Growth_3y_mean', 'Importer_Demand_Growth_3y_std',
    'Algeria_Share_in_Importer_3y_mean', 'Algeria_Share_in_Importer_3y_std',
    'market_opp_score_3y_mean', 'market_opp_score_3y_std',
    # Target encoding
    'hs2_target_enc', 'imp_target_enc', 'imp_hs2_target_enc',
    # Global context
    'global_demand_growth', 'n_importers',
    # Interactions + trajectory
    'imp_pct_x_growth', 'market_opp_score_diff_2y',
    # Sector + year
    'in_priority_hs2', 'Year',
] + imp_macro_cols
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]

X = df[FEATURE_COLS].astype('float32').copy()
X = X.replace([np.inf, -np.inf], np.nan).fillna(X.median(numeric_only=True)).astype('float32')

y = df['label'].astype('int8').values
label_year = df['label_year'].astype('int16').values
importer = df['Importer_ISO3'].astype(str).values
product = df['Product'].astype(str).values
in_margin = df['in_margin'].values

print(f'X shape: {X.shape}  features: {len(X.columns)}  memory: {X.memory_usage(deep=True).sum()/1e9:.2f} GB')

X shape: (6677951, 51)  features: 51  memory: 1.36 GB


## 15. Train / test split

**Purpose and approach**
This cell performs the time-based train/test split and standardizes features.
If margin exclusion is enabled, ambiguous boundary rows are removed from training to improve label quality.

In [16]:
from sklearn.preprocessing import StandardScaler

mask_tr_full = label_year <= TRAIN_MAX_LABEL_YEAR
mask_te = label_year == TEST_LABEL_YEAR

if EXCLUDE_MARGIN:
    mask_tr = mask_tr_full & ~in_margin
    print(f'Margin exclusion ON: removed {int(mask_tr_full.sum()-mask_tr.sum()):,} samples')
else:
    mask_tr = mask_tr_full

X_tr_df, X_te_df = X.loc[mask_tr], X.loc[mask_te]
y_tr, y_te = y[mask_tr], y[mask_te]

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr_df.values).astype('float32')
X_te_s = scaler.transform(X_te_df.values).astype('float32')

print(f'\nTrain: {X_tr_df.shape}  Test: {X_te_df.shape}')
print('Train class balance:', dict(zip(*np.unique(y_tr, return_counts=True))))
print('Test  class balance:', dict(zip(*np.unique(y_te, return_counts=True))))

Margin exclusion ON: removed 450,603 samples

Train: (5631076, 51)  Test: (596272, 51)
Train class balance: {np.int8(0): np.int64(1932563), np.int8(1): np.int64(1797975), np.int8(2): np.int64(1900538)}
Test  class balance: {np.int8(0): np.int64(198757), np.int8(1): np.int64(198758), np.int8(2): np.int64(198757)}


## 16. Persistence baseline

**Purpose and approach**
This cell computes the persistence baseline by predicting that next year's class equals this year's class.
It is a strong benchmark in time-series settings and must be exceeded by any model to show value.

In [17]:
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

y_baseline = df.loc[mask_te, 'label_thisyear'].astype('int8').values
f1_b = f1_score(y_te, y_baseline, average='macro')
acc_b = accuracy_score(y_te, y_baseline)
print(f'Persistence baseline:  macro_F1={f1_b:.4f}  acc={acc_b:.4f}')
print(classification_report(y_te, y_baseline, target_names=['Low','Medium','High'], zero_division=0))
print('confusion:'); print(confusion_matrix(y_te, y_baseline))

Persistence baseline:  macro_F1=0.4440  acc=0.4444
              precision    recall  f1-score   support

         Low       0.45      0.43      0.44    198757
      Medium       0.35      0.36      0.36    198758
        High       0.53      0.54      0.54    198757

    accuracy                           0.44    596272
   macro avg       0.44      0.44      0.44    596272
weighted avg       0.44      0.44      0.44    596272

confusion:
[[ 85659  73731  39367]
 [ 69968  71330  57460]
 [ 34897  55893 107967]]


## 17. XGBoost hyperparameter tuning on a subsample

**Purpose and approach**
This cell runs a time-aware hyperparameter search for XGBoost on a stratified subsample. The split respects label-year order, which avoids future leakage.
The subsample keeps tuning feasible while preserving class balance.

In [18]:
import xgboost as xgb
BEST_XGB_PARAMS = None

if RUN_TUNING:
    from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
    TUNE_N = 500_000
    rng = np.random.default_rng(RANDOM_STATE)
    classes_tr = np.unique(y_tr)
    per_cls = TUNE_N // len(classes_tr)
    idx_per_class = []
    for c in classes_tr:
        ci = np.where(y_tr == c)[0]
        idx_per_class.append(rng.choice(ci, size=min(per_cls, ci.size), replace=False))
    tune_idx = np.concatenate(idx_per_class)
    tune_idx = tune_idx[np.argsort(label_year[mask_tr][tune_idx])]
    X_tune = X_tr_df.iloc[tune_idx]; y_tune = y_tr[tune_idx]

    base = xgb.XGBClassifier(
        objective='multi:softprob', num_class=3,
        tree_method='hist', device=XGB_DEVICE,
        eval_metric='mlogloss', n_jobs=-1, random_state=RANDOM_STATE,
    )
    param_dist = {
        'n_estimators':      [500, 800, 1200],
        'max_depth':         [6, 8, 10, 12],
        'learning_rate':     [0.03, 0.05, 0.08, 0.1],
        'subsample':         [0.7, 0.85, 1.0],
        'colsample_bytree':  [0.7, 0.85, 1.0],
        'min_child_weight':  [1, 5, 20],
        'reg_alpha':         [0.0, 0.1, 0.5],
        'reg_lambda':        [1.0, 5.0, 10.0],
    }
    search = RandomizedSearchCV(base, param_distributions=param_dist, n_iter=20,
                                scoring='f1_macro', cv=TimeSeriesSplit(n_splits=3),
                                n_jobs=1, verbose=1, random_state=RANDOM_STATE)
    search.fit(X_tune.values, y_tune)
    BEST_XGB_PARAMS = search.best_params_
    print('Best XGB params:', BEST_XGB_PARAMS)
    print('Best CV macro_F1:', round(search.best_score_, 4))
else:
    print('Tuning skipped (RUN_TUNING=False).')

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best XGB params: {'subsample': 0.85, 'reg_lambda': 10.0, 'reg_alpha': 0.0, 'n_estimators': 1200, 'min_child_weight': 20, 'max_depth': 8, 'learning_rate': 0.03, 'colsample_bytree': 0.7}
Best CV macro_F1: 0.5837


## 18. Train models

**Purpose and approach**
This cell defines the evaluation setup for model training. We keep one shared interface for all models so that results are directly comparable.
The goal is to evaluate each model on the same test year using the same metrics.

In [19]:
results = {'Persistence': {'macro_f1': float(f1_b), 'accuracy': float(acc_b)}}
predictions = {'Persistence': {'pred': y_baseline, 'proba': None}}
fitted_models = {}

def evaluate(name, model, X_eval, y_eval, proba=None):
    if proba is None and hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X_eval)
    yhat = (np.argmax(proba, axis=1) if proba is not None else model.predict(X_eval)).astype('int8')
    f1 = f1_score(y_eval, yhat, average='macro')
    acc = accuracy_score(y_eval, yhat)
    results[name] = {'macro_f1': float(f1), 'accuracy': float(acc)}
    predictions[name] = {'pred': yhat, 'proba': proba}
    print(f'\n=== {name} ===  macro_F1={f1:.4f}  acc={acc:.4f}')
    print(classification_report(y_eval, yhat, target_names=['Low','Medium','High'], zero_division=0))
    print('confusion:'); print(confusion_matrix(y_eval, yhat))

### 18.1 Random Forest

**Purpose and approach**
This cell trains the Random Forest baseline. Random Forests are strong on heterogeneous tabular data and help validate that the feature set has predictive value.
We keep the model size moderate to balance performance and runtime.

In [20]:
if RUN_RF:
    try:
        from sklearn.ensemble import RandomForestClassifier
        rf = RandomForestClassifier(n_estimators=300, max_depth=16, min_samples_leaf=60,
                                     n_jobs=-1, random_state=RANDOM_STATE)
        rf.fit(X_tr_df.values, y_tr)
        evaluate('RandomForest', rf, X_te_df.values, y_te)
        fitted_models['RandomForest'] = rf
    except Exception as e:
        print('RandomForest failed:', e); rf = None
else:
    rf = None; print('RF skipped')

KeyboardInterrupt: 

### 18.2 XGBoost (tuned if available)

**Purpose and approach**
This cell trains XGBoost (using tuned parameters if available) and evaluates it on the test year.
XGBoost is a strong boosted-tree model that captures non-linear interactions in trade data.

In [21]:
if RUN_XGB:
    try:
        xgb_kwargs = dict(
            objective='multi:softprob', num_class=3,
            tree_method='hist', device=XGB_DEVICE,
            n_jobs=-1, eval_metric='mlogloss', random_state=RANDOM_STATE,
            n_estimators=800, max_depth=8, learning_rate=0.06,
            subsample=0.8, colsample_bytree=0.8,
        )
        if BEST_XGB_PARAMS is not None:
            xgb_kwargs.update(BEST_XGB_PARAMS)
            print('Using tuned params:', BEST_XGB_PARAMS)
        xgb_clf = xgb.XGBClassifier(**xgb_kwargs)
        xgb_clf.fit(X_tr_df.values, y_tr, verbose=False)
        evaluate('XGBoost', xgb_clf, X_te_df.values, y_te)
        fitted_models['XGBoost'] = xgb_clf
    except Exception as e:
        print('XGBoost failed:', e); xgb_clf = None
else:
    xgb_clf = None; print('XGB skipped')

Using tuned params: {'subsample': 0.85, 'reg_lambda': 10.0, 'reg_alpha': 0.0, 'n_estimators': 1200, 'min_child_weight': 20, 'max_depth': 8, 'learning_rate': 0.03, 'colsample_bytree': 0.7}

=== XGBoost ===  macro_F1=0.5844  acc=0.5965
              precision    recall  f1-score   support

         Low       0.59      0.77      0.66    198757
      Medium       0.52      0.36      0.43    198758
        High       0.66      0.66      0.66    198757

    accuracy                           0.60    596272
   macro avg       0.59      0.60      0.58    596272
weighted avg       0.59      0.60      0.58    596272

confusion:
[[152443  29909  16405]
 [ 75813  72161  50784]
 [ 32080  35624 131053]]


### 18.3 LightGBM

**Purpose and approach**
This cell trains LightGBM with GPU support and evaluates it on the test year.
LightGBM is efficient on large datasets and provides a strong alternative to XGBoost.

In [22]:
if RUN_LGBM:
    try:
        import lightgbm as lgb
        lgb_kwargs = dict(n_estimators=1000, max_depth=-1, num_leaves=127, learning_rate=0.05,
                          subsample=0.85, colsample_bytree=0.85, objective='multiclass',
                          n_jobs=-1, random_state=RANDOM_STATE, device=LGB_DEVICE,
                          min_data_in_leaf=50)
        try:
            lgb_clf = lgb.LGBMClassifier(**lgb_kwargs)
            lgb_clf.fit(X_tr_df.values, y_tr)
        except lgb.basic.LightGBMError as e:
            print('LightGBM GPU failed, fallback to CPU:', e)
            lgb_kwargs['device'] = 'cpu'
            lgb_clf = lgb.LGBMClassifier(**lgb_kwargs)
            lgb_clf.fit(X_tr_df.values, y_tr)
        evaluate('LightGBM', lgb_clf, X_te_df.values, y_te)
        fitted_models['LightGBM'] = lgb_clf
    except Exception as e:
        print('LightGBM failed entirely:', e); lgb_clf = None
else:
    lgb_clf = None; print('LGBM skipped')

[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 9462
[LightGBM] [Info] Number of data points in the train set: 5631076, number of used features: 51
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 33 dense feature groups (193.33 MB) transferred to GPU in 0.182895 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score -1.069453
[LightGBM] [Info] Start training from score -1.141640
[LightGBM] [Info] Start training from score -1.086164
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50

=== LightGBM ===  macro_F1=0.5839  acc=0.5957
              precision    recall  f1-score   support

         Low       0.59      0.77      0.66    198757
      Medium       0.52      0.36      0.43    198758
        High       0.66      0.66      0.66    198757

    accuracy                           0.60    596272
   macro avg       0.59      0.60      0.58    596272
weighted avg       0.59      0.60      0.58    596272

confusion:
[[152089  30024  16644]
 [ 75598  72377  50783]
 [ 32084  35925 13

### 18.4 CatBoost (native categorical handling)

CatBoost handles categorical features natively without one-hot encoding — uses target statistics computed on permuted data to avoid leakage. Often outperforms XGBoost/LightGBM on data with high-cardinality categoricals like Importer (227 levels) × HS2 (96 levels).

**Purpose and approach**
This cell trains CatBoost, which can model categorical effects without manual encoding. This is useful for high-cardinality features like importer and HS2 sector.
We evaluate it alongside the other models to see whether native categorical handling improves performance.

In [23]:
if RUN_CATBOOST:
    try:
        from catboost import CatBoostClassifier, Pool
        cat_clf = CatBoostClassifier(
            iterations=1000, depth=8, learning_rate=0.05,
            loss_function='MultiClass', random_seed=RANDOM_STATE,
            task_type=CB_TASK, devices='0:1' if GPU_AVAILABLE else None,
            verbose=False,
        )
        cat_clf.fit(X_tr_df.values, y_tr)
        cb_proba = cat_clf.predict_proba(X_te_df.values)
        evaluate('CatBoost', cat_clf, X_te_df.values, y_te, proba=cb_proba)
        fitted_models['CatBoost'] = cat_clf
    except Exception as e:
        print('CatBoost failed:', e); cat_clf = None
else:
    cat_clf = None; print('CatBoost skipped')


=== CatBoost ===  macro_F1=0.5822  acc=0.5942
              precision    recall  f1-score   support

         Low       0.58      0.77      0.66    198757
      Medium       0.52      0.36      0.43    198758
        High       0.66      0.65      0.66    198757

    accuracy                           0.59    596272
   macro avg       0.59      0.59      0.58    596272
weighted avg       0.59      0.59      0.58    596272

confusion:
[[152420  30419  15918]
 [ 76186  71799  50773]
 [ 32586  36073 130098]]


### 18.5 Linear SVM

**Purpose and approach**
This cell trains a calibrated Linear SVM on a stratified subsample. The linear model provides a contrast to tree-based methods and yields probability scores after calibration.
It helps us assess how much of the signal is linearly separable.

In [24]:
if RUN_SVM:
    try:
        from sklearn.svm import LinearSVC
        from sklearn.calibration import CalibratedClassifierCV
        SVM_SAMPLE = 200_000
        rng = np.random.default_rng(RANDOM_STATE)
        classes_tr = np.unique(y_tr)
        per_cls = SVM_SAMPLE // len(classes_tr)
        idx_per_class = []
        for c in classes_tr:
            ci = np.where(y_tr == c)[0]
            idx_per_class.append(rng.choice(ci, size=min(per_cls, ci.size), replace=False))
        svm_idx = np.concatenate(idx_per_class)
        svm = CalibratedClassifierCV(LinearSVC(C=0.5, dual=False, max_iter=4000), cv=3)
        svm.fit(X_tr_s[svm_idx], y_tr[svm_idx])
        evaluate('LinearSVM', svm, X_te_s, y_te)
        fitted_models['LinearSVM'] = svm
    except Exception as e:
        print('LinearSVM failed:', e); svm = None
else:
    svm = None; print('SVM skipped')


=== LinearSVM ===  macro_F1=0.5491  acc=0.5656
              precision    recall  f1-score   support

         Low       0.57      0.71      0.63    198757
      Medium       0.46      0.30      0.36    198758
        High       0.62      0.69      0.65    198757

    accuracy                           0.57    596272
   macro avg       0.55      0.57      0.55    596272
weighted avg       0.55      0.57      0.55    596272

confusion:
[[140560  37424  20773]
 [ 75004  58963  64791]
 [ 29241  31768 137748]]


### 18.6 Keras MLP with label smoothing


**Purpose and approach**
This cell trains the Keras MLP with label smoothing and evaluates it. Label smoothing helps the model avoid overconfidence at class boundaries.
The neural model provides a different inductive bias and can pick up interactions that trees might miss.

In [25]:
if RUN_MLP:
    try:
        from tensorflow.keras import layers, models, callbacks
        tf.keras.utils.set_random_seed(RANDOM_STATE)
        mlp = models.Sequential([
            layers.Input(shape=(X_tr_s.shape[1],)),
            layers.Dense(256, activation='relu'), layers.BatchNormalization(), layers.Dropout(0.3),
            layers.Dense(128, activation='relu'), layers.BatchNormalization(), layers.Dropout(0.3),
            layers.Dense(64,  activation='relu'),
            layers.Dense(3, activation='softmax'),
        ])
        # Categorical cross-entropy with label smoothing for the Medium boundary
        loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=MLP_LABEL_SMOOTHING)
        mlp.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=loss, metrics=['accuracy'])
        y_tr_oh = tf.keras.utils.to_categorical(y_tr, num_classes=3)
        y_te_oh = tf.keras.utils.to_categorical(y_te, num_classes=3)
        early = callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)
        mlp.fit(X_tr_s, y_tr_oh, validation_split=0.1, batch_size=4096, epochs=30,
                callbacks=[early], verbose=2)
        evaluate('Keras MLP', mlp, X_te_s, y_te,
                 proba=mlp.predict(X_te_s, batch_size=8192, verbose=0))
        fitted_models['Keras MLP'] = mlp
    except Exception as e:
        print('MLP failed:', e); mlp = None
else:
    mlp = None; print('MLP skipped')

I0000 00:00:1779723707.370183      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13419 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779723707.383480      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13755 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/30


I0000 00:00:1779723714.094859     225 service.cc:152] XLA service 0x78763b053640 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779723714.094911     225 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1779723714.094916     225 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1779723714.601216     225 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1779723717.473537     225 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1238/1238 - 15s - 12ms/step - accuracy: 0.5925 - loss: 0.8934 - val_accuracy: 0.6003 - val_loss: 0.8792
Epoch 2/30
1238/1238 - 5s - 4ms/step - accuracy: 0.6052 - loss: 0.8751 - val_accuracy: 0.6025 - val_loss: 0.8765
Epoch 3/30
1238/1238 - 5s - 4ms/step - accuracy: 0.6072 - loss: 0.8723 - val_accuracy: 0.6027 - val_loss: 0.8755
Epoch 4/30
1238/1238 - 5s - 4ms/step - accuracy: 0.6085 - loss: 0.8705 - val_accuracy: 0.6037 - val_loss: 0.8745
Epoch 5/30
1238/1238 - 5s - 4ms/step - accuracy: 0.6094 - loss: 0.8694 - val_accuracy: 0.6039 - val_loss: 0.8746
Epoch 6/30
1238/1238 - 5s - 4ms/step - accuracy: 0.6099 - loss: 0.8686 - val_accuracy: 0.6041 - val_loss: 0.8746
Epoch 7/30
1238/1238 - 5s - 4ms/step - accuracy: 0.6106 - loss: 0.8679 - val_accuracy: 0.6038 - val_loss: 0.8746
Epoch 8/30
1238/1238 - 5s - 4ms/step - accuracy: 0.6109 - loss: 0.8674 - val_accuracy: 0.6040 - val_loss: 0.8745

=== Keras MLP ===  macro_F1=0.5782  acc=0.5903
              precision    recall  f1-score   support

  

## 19. Weighted ensemble of tree models

**Purpose and approach**
This cell builds a weighted ensemble of the tree models based on their macro-F1 scores. Probability averaging can improve stability and reduce model-specific noise.
The ensemble is optional and only runs if multiple tree models succeed.

In [26]:
ensemble_candidates = [k for k in ['XGBoost', 'LightGBM', 'RandomForest', 'CatBoost']
                       if k in predictions and predictions[k]['proba'] is not None]
if len(ensemble_candidates) >= 2:
    raw_w = {k: results[k]['macro_f1'] for k in ensemble_candidates}
    weights = {k: v / sum(raw_w.values()) for k, v in raw_w.items()}
    print('Ensemble weights:', {k: round(v, 3) for k, v in weights.items()})
    ens_proba = sum(weights[k] * predictions[k]['proba'] for k in ensemble_candidates)
    ens_pred = np.argmax(ens_proba, axis=1).astype('int8')
    f1_e = f1_score(y_te, ens_pred, average='macro')
    acc_e = accuracy_score(y_te, ens_pred)
    results['Ensemble'] = {'macro_f1': float(f1_e), 'accuracy': float(acc_e)}
    predictions['Ensemble'] = {'pred': ens_pred, 'proba': ens_proba}
    print(f'\n=== Ensemble ===  macro_F1={f1_e:.4f}  acc={acc_e:.4f}')
    print(classification_report(y_te, ens_pred, target_names=['Low','Medium','High'], zero_division=0))
else:
    print(f'Only {len(ensemble_candidates)} model(s) succeeded — skipping ensemble.')

Ensemble weights: {'XGBoost': 0.334, 'LightGBM': 0.334, 'CatBoost': 0.333}

=== Ensemble ===  macro_F1=0.5843  acc=0.5964
              precision    recall  f1-score   support

         Low       0.59      0.77      0.66    198757
      Medium       0.52      0.36      0.43    198758
        High       0.66      0.66      0.66    198757

    accuracy                           0.60    596272
   macro avg       0.59      0.60      0.58    596272
weighted avg       0.59      0.60      0.58    596272



## 20. Leaderboard + feature importance

**Purpose and approach**
This cell writes the leaderboard to disk after all models finish.
Keeping a saved table supports reporting and later auditing of model performance.

In [27]:
leaderboard = pd.DataFrame(results).T.sort_values('macro_f1', ascending=False).round(4)
print(leaderboard)
leaderboard.to_csv(OUT_DIR / 'market_classification_leaderboard.csv')

tree_candidates = {k: v for k, v in fitted_models.items() if k in ('XGBoost','LightGBM','RandomForest','CatBoost')}
if tree_candidates:
    best_tree = max(tree_candidates, key=lambda k: results[k]['macro_f1'])
    m = tree_candidates[best_tree]
    imp = getattr(m, 'feature_importances_', None)
    if imp is None and hasattr(m, 'get_feature_importance'):
        imp = m.get_feature_importance()
    if imp is not None:
        fi = pd.Series(imp, index=X_tr_df.columns).sort_values(ascending=False)
        print(f'\nTop 25 features ({best_tree}):')
        print(fi.head(25).round(4))

             macro_f1  accuracy
XGBoost        0.5844    0.5965
Ensemble       0.5843    0.5964
LightGBM       0.5839    0.5957
CatBoost       0.5822    0.5942
Keras MLP      0.5782    0.5903
LinearSVM      0.5491    0.5656
Persistence    0.4440    0.4444

Top 25 features (XGBoost):
imp_hs2_target_enc               0.2895
in_priority_hs2                  0.1418
Importer_Demand_Pct              0.0709
market_opp_score_3y_mean         0.0686
Importer_Demand_Pct_lag1         0.0625
Importer_Demand_Growth           0.0310
Importer_Demand_Growth_3y_mean   0.0238
label_thisyear                   0.0196
log_Importer_Demand              0.0175
Importer_Demand_3y_mean          0.0173
hs2_target_enc                   0.0167
imp_pct_x_growth                 0.0154
market_opp_score                 0.0144
Importer_Demand_Growth_3y_std    0.0136
label_lag1                       0.0127
imp_target_enc                   0.0113
log_Global_Demand                0.0096
is_eu                            0.0

## 21. HS code dictionary

**Purpose and approach**
This cell loads HS chapter names and optional HS6 descriptions. It converts product codes into human-readable labels used in the final ranking output.
Readable labels are essential for stakeholder interpretation of the top opportunities.

In [28]:
HS_CHAPTER_NAMES = {
    '01': 'Live animals', '02': 'Meat', '03': 'Fish/crustaceans', '04': 'Dairy/eggs/honey',
    '05': 'Other animal products', '06': 'Live trees/plants/flowers',
    '07': 'Edible vegetables', '08': 'Edible fruit/nuts',
    '09': 'Coffee/tea/spices', '10': 'Cereals', '11': 'Milling products',
    '12': 'Oil seeds/medicinal plants', '13': 'Lac/gums/resins', '14': 'Vegetable plaiting',
    '15': 'Animal/vegetable fats and oils',
    '16': 'Meat/fish preparations', '17': 'Sugars/confectionery', '18': 'Cocoa',
    '19': 'Cereal/flour/milk preparations, pastry',
    '20': 'Vegetable/fruit preparations', '21': 'Miscellaneous edible preparations',
    '22': 'Beverages/spirits/vinegar', '23': 'Food industry residues', '24': 'Tobacco',
    '25': 'Salt/sulphur/earth/stone (construction)',
    '26': 'Ores/slag/ash', '27': 'Mineral fuels (oil, gas, coal)',
    '28': 'Inorganic chemicals', '29': 'Organic chemicals', '30': 'Pharmaceuticals',
    '31': 'Fertilizers', '32': 'Tanning/dyeing', '33': 'Essential oils/cosmetics',
    '34': 'Soap/waxes', '35': 'Albuminoids/starches', '36': 'Explosives',
    '37': 'Photographic goods', '38': 'Misc chemicals', '39': 'Plastics', '40': 'Rubber',
    '41': 'Raw hides', '42': 'Leather', '43': 'Furskins',
    '44': 'Wood', '45': 'Cork', '46': 'Straw manufactures', '47': 'Wood pulp',
    '48': 'Paper', '49': 'Printed books',
    '50': 'Silk', '51': 'Wool', '52': 'Cotton', '53': 'Other vegetable textile',
    '54': 'Man-made filaments', '55': 'Man-made staple', '56': 'Wadding/felt',
    '57': 'Carpets', '58': 'Special woven fabrics', '59': 'Coated textiles',
    '60': 'Knitted fabrics', '61': 'Apparel — knitted', '62': 'Apparel — not knitted',
    '63': 'Other textile articles', '64': 'Footwear', '65': 'Headgear', '66': 'Umbrellas',
    '67': 'Prepared feathers',
    '68': 'Articles of stone/plaster/cement', '69': 'Ceramics', '70': 'Glass',
    '71': 'Pearls/precious stones/metals', '72': 'Iron and steel',
    '73': 'Articles of iron/steel', '74': 'Copper', '75': 'Nickel', '76': 'Aluminium',
    '78': 'Lead', '79': 'Zinc', '80': 'Tin', '81': 'Other base metals',
    '82': 'Tools/cutlery', '83': 'Misc articles of base metal',
    '84': 'Machinery/mechanical', '85': 'Electrical machinery', '86': 'Railway vehicles',
    '87': 'Vehicles other than railway', '88': 'Aircraft/spacecraft', '89': 'Ships/boats',
    '90': 'Optical/medical/precision', '91': 'Clocks/watches', '92': 'Musical instruments',
    '93': 'Arms/ammunition', '94': 'Furniture/lamps', '95': 'Toys/games',
    '96': 'Misc manufactured', '97': 'Works of art', '99': 'Commodities n.e.s.',
}
def hs_chapter_label(hs2):
    name = HS_CHAPTER_NAMES.get(str(hs2).zfill(2), 'Unknown')
    star = ' ★' if str(hs2).zfill(2) in PRIORITY_HS2 else ''
    return f'HS{hs2} — {name}{star}'
hs6_lookup = {}
for name in ['product_codes_HS12_V202601.csv', 'product_codes_HS22_V202601.csv']:
    p = find_file(name)
    if p is not None:
        try:
            pc = pd.read_csv(p)
            code_col = next((c for c in ['code','HS6','product_code'] if c in pc.columns), pc.columns[0])
            desc_col = next((c for c in ['description','desc','name'] if c in pc.columns), pc.columns[-1])
            hs6_lookup = dict(zip(pc[code_col].astype(str).str.zfill(6), pc[desc_col].astype(str)))
            print(f'Loaded {len(hs6_lookup):,} HS6 descriptions from {p.name}')
            break
        except Exception as e:
            print(f'Could not parse {p.name}: {e}')
if not hs6_lookup:
    print('HS6 descriptions not found — falling back to HS chapter names.')
def hs6_description(code):
    return hs6_lookup.get(str(code).zfill(6), '')

HS6 descriptions not found — falling back to HS chapter names.


## 22. Algeria market ranking — the deliverable

**Purpose and approach**
This cell builds the Algeria market ranking and saves `market_opportunity_ranking.csv`. It combines predictions, importer demand signals, and product descriptions.
This is the main deliverable: a ranked list of target countries for each product.

In [29]:
candidates = [k for k in results if k != 'Persistence']
if not candidates:
    print('No ML model succeeded — cannot produce ranking.')
else:
    best_name = max(candidates, key=lambda k: results[k]['macro_f1'])
    best_proba = predictions[best_name]['proba']
    best_pred = predictions[best_name]['pred']
    print(f'Best ML model: {best_name}  macro_F1={results[best_name]["macro_f1"]:.4f}'
          f'  (vs persistence={results["Persistence"]["macro_f1"]:.4f})')

    ranking = pd.DataFrame({
        'label_year': label_year[mask_te],
        'feature_year': label_year[mask_te] - 1,
        'Importer_ISO3': importer[mask_te],
        'Product': product[mask_te],
        'Importer_Demand_T': df.loc[mask_te, 'Importer_Demand'].values,
        'Importer_Demand_Pct_T': df.loc[mask_te, 'Importer_Demand_Pct'].values,
        'Importer_Demand_Growth_T': df.loc[mask_te, 'Importer_Demand_Growth'].values,
        'Algeria_Value_T': df.loc[mask_te, 'Algeria_Value'].values,
        'Algeria_Share_in_Importer_T': df.loc[mask_te, 'Algeria_Share_in_Importer'].values,
        'Algeria_HS2_to_Importer_T': df.loc[mask_te, 'Algeria_HS2_to_Importer'].values,
        'has_trade_relationship': df.loc[mask_te, 'has_trade_relationship'].values,
        'is_eu': df.loc[mask_te, 'is_eu'].values,
        'is_francophone': df.loc[mask_te, 'is_francophone'].values,
        'class_T': df.loc[mask_te, 'label_thisyear'].map(INV_LABEL).values,
        'pred_label': [INV_LABEL[int(p)] for p in best_pred],
    })
    if best_proba is not None:
        ranking['prob_Low']    = best_proba[:, 0]
        ranking['prob_Medium'] = best_proba[:, 1]
        ranking['prob_High']   = best_proba[:, 2]
    ranking['HS2'] = ranking['Product'].str.zfill(6).str[:2]
    ranking['HS_Chapter'] = ranking['HS2'].map(hs_chapter_label)
    ranking['HS6_Description'] = ranking['Product'].map(hs6_description)
    ranking['Description'] = ranking['HS6_Description'].where(ranking['HS6_Description'] != '', ranking['HS_Chapter'])

    sort_keys = ['prob_High' if 'prob_High' in ranking.columns else 'pred_label', 'Importer_Demand_T']
    ranking = ranking.sort_values(sort_keys, ascending=False).reset_index(drop=True)
    out_path = OUT_DIR / 'market_opportunity_ranking.csv'
    ranking.to_csv(out_path, index=False)
    print(f'Total rows: {len(ranking):,}   Saved: {out_path}')
    print(f'Predicted High: {(ranking["pred_label"]=="High").sum():,}   '
          f'Medium: {(ranking["pred_label"]=="Medium").sum():,}   '
          f'Low: {(ranking["pred_label"]=="Low").sum():,}')

Best ML model: XGBoost  macro_F1=0.5844  (vs persistence=0.4440)
Total rows: 596,272   Saved: /kaggle/working/market_opportunity_ranking.csv
Predicted High: 198,242   Medium: 137,694   Low: 260,336


**How to interpret the top 20 markets**
The top rows represent importer-product pairs where the model is most confident that Algeria has a strong opportunity next year. Use `prob_High` as the confidence score and `Importer_Demand_T` to size the market.
Low `Algeria_Share_in_Importer_T` combined with high confidence suggests a clear expansion gap. Use the EU/francophone flags and product description to assess feasibility and market access.

### Top 20 (Importer, Product) opportunities — human-readable

**Purpose and approach**
This cell prints a human-readable top-20 table and a short narrative summary. The aim is to translate model scores into a list that decision makers can act on.
It uses product descriptions and importer context so the output reads like a market short-list, not raw model output.

In [30]:
top20 = ranking[ranking['pred_label'] == 'High'].head(20).copy()
display_cols = ['Importer_ISO3', 'Product', 'Description', 'prob_High',
                'Importer_Demand_T', 'Importer_Demand_Pct_T', 'Algeria_Share_in_Importer_T',
                'Algeria_HS2_to_Importer_T', 'is_eu', 'is_francophone']
print('Top 20 predicted-High (Importer, Product) opportunities for Algeria (2023):\n')
with pd.option_context('display.max_colwidth', 60):
    print(top20[display_cols].to_string(index=False))

print('\n' + '=' * 70)
print('NARRATIVE SUMMARY (top 10)')
print('=' * 70)
for i, row in top20.head(10).iterrows():
    code = row['Product']; desc = row['Description']
    imp = row['Importer_ISO3']
    pct = row['prob_High'] * 100 if pd.notna(row.get('prob_High')) else 0
    demand_m = (row['Importer_Demand_T'] or 0) / 1000
    share = row['Algeria_Share_in_Importer_T']
    hs2_alg_k = (row['Algeria_HS2_to_Importer_T'] or 0) / 1
    tags = []
    if row['is_eu']: tags.append('EU')
    if row['is_francophone']: tags.append('Francophone')
    tag_str = f' [{", ".join(tags)}]' if tags else ''
    print(f'\n#{i+1}  Sell HS {code} ({desc}) to {imp}{tag_str}')
    print(f'      Predicted: High opportunity (confidence {pct:.1f}%)')
    print(f'      Importer buys ~{demand_m:,.1f} M USD/year; Algeria currently captures {share:.4f}%')
    print(f'      Algeria already sells ~{hs2_alg_k:,.1f} K USD in this HS chapter to {imp}')

Top 20 predicted-High (Importer, Product) opportunities for Algeria (2023):

Importer_ISO3 Product                                     Description  prob_High  Importer_Demand_T  Importer_Demand_Pct_T  Algeria_Share_in_Importer_T  Algeria_HS2_to_Importer_T  is_eu  is_francophone
          DEU   81110                      HS08 — Edible fruit/nuts ★     0.9999       148,600.8125                10.2911                       0.0000                 3,080.5759      1               0
          DEU   71090                      HS07 — Edible vegetables ★     0.9999       111,339.2891                12.0343                       0.0000                     0.0140      1               0
          USA  680520       HS68 — Articles of stone/plaster/cement ★     0.9999       149,443.7969                11.1204                       0.0000                     0.0000      0               0
          DEU  210500      HS21 — Miscellaneous edible preparations ★     0.9999       492,931.1875                

### Aggregate by importer

**Purpose and approach**
This cell aggregates the ranking by importer to highlight which countries appear most often in the High opportunity list.
This provides a country-level perspective to complement the product-level ranking.

In [31]:
imp_summary = (ranking[ranking['pred_label'] == 'High']
               .groupby('Importer_ISO3')
               .agg(n_products_High=('Product', 'count'),
                    total_demand_M_USD=('Importer_Demand_T', lambda s: round(s.sum()/1000, 1)),
                    mean_prob_High=('prob_High', 'mean'),
                    is_eu=('is_eu', 'max'),
                    is_francophone=('is_francophone', 'max'))
               .sort_values('total_demand_M_USD', ascending=False).head(20))
print('Top 20 importer countries by High-opportunity product count and demand size:\n')
print(imp_summary.to_string())

Top 20 importer countries by High-opportunity product count and demand size:

               n_products_High  total_demand_M_USD  mean_prob_High  is_eu  is_francophone
Importer_ISO3                                                                            
USA                       3611      3,053,206.5000          0.9086      0               0
CHN                       3275      1,990,727.3750          0.7701      0               0
DEU                       3640      1,401,784.1250          0.9022      1               0
JPN                       3313        781,823.1875          0.7927      0               0
GBR                       3398        724,887.0000          0.8146      0               0
FRA                       3523        722,947.3125          0.8473      1               1
NLD                       3461        657,675.1250          0.8018      1               0
ITA                       3443        620,390.1875          0.8228      1               0
KOR                   

## 23. Persist artifacts

**Purpose and approach**
This cell saves the fitted models, scaler, and feature list for future scoring.
Persisting artifacts enables reproducible deployment without retraining.

In [32]:
import joblib
joblib.dump(scaler, OUT_DIR / 'market_scaler.joblib')
if 'XGBoost' in fitted_models:      xgb_clf.save_model(str(OUT_DIR / 'market_xgb.json'))
if 'LightGBM' in fitted_models:     lgb_clf.booster_.save_model(str(OUT_DIR / 'market_lgbm.txt'))
if 'RandomForest' in fitted_models: joblib.dump(rf, OUT_DIR / 'market_rf.joblib')
if 'CatBoost' in fitted_models:     cat_clf.save_model(str(OUT_DIR / 'market_catboost.cbm'))
if 'Keras MLP' in fitted_models:    mlp.save(OUT_DIR / 'market_mlp.keras')
with open(OUT_DIR / 'market_feature_columns.json', 'w') as f:
    json.dump(list(X_tr_df.columns), f)
print('Artifacts saved to', OUT_DIR)

Artifacts saved to /kaggle/working


## How to use the saved models (local path)

The artifacts in `model/classification/market_classification` are ready for inference when the same feature engineering steps are reproduced.
Minimum inputs for scoring are the feature columns used in training.

**To score new data locally:**
- Load `market_feature_columns.json` and build the feature matrix with the same columns in the same order.
- Load `market_scaler.joblib` and apply `scaler.transform()` to the feature matrix.
- Load the desired model file (e.g., `market_xgb.json`, `market_lgbm.txt`, `market_rf.joblib`, `market_catboost.cbm`, or `market_mlp.keras`).
- Run `predict_proba()` (or equivalent) to get `prob_Low`, `prob_Medium`, `prob_High`, then map `argmax` to labels.

For an ensemble, average probabilities from multiple tree models using their macro-F1 weights (the same approach used in this notebook).

## Interpreting the results (and why they are still useful)

The model performance is moderate because the task is inherently noisy: bilateral trade relationships shift, and Algeria has zero exports in most importer-product pairs.
Even with modest accuracy, the ranking is still valuable because it concentrates attention on the highest-confidence, highest-demand opportunities.

Practical ways this helps decision-making:
- It narrows the search space from hundreds of thousands of pairs to a short list for outreach or validation.
- It flags high-demand markets where Algeria’s share is low, which are actionable expansion gaps.
- It provides a consistent, repeatable baseline that can be tracked year-over-year to measure progress.

## Summary

This notebook builds a market opportunity ranking that is grounded in importer demand, Algeria’s current position, and sector priorities.
The saved artifacts allow the same pipeline to score future data without retraining, while the ranking provides a practical shortlist for export strategy and market outreach.